# 11.1 Spark Performance Fundamentals

## Learning Objectives

By the end of this lesson you will understand:
- Why Spark jobs become slow
- Common Spark execution bottlenecks
- CPU, Memory, Disk, and Network bottlenecks
- Why optimization matters before scaling infrastructure

> **Core Rule:** Spark is fast, but Spark jobs can still become slow. Before fixing performance, we must understand where time is actually being spent.

This lesson builds the foundation for all advanced Spark optimization topics.

## Setup: Initialize Spark
Before we discuss bottlenecks, let's start our local Spark Session so we can demonstrate these concepts in code.

In [ ]:
from pyspark.sql import SparkSession
import pyspark.sql.functions as F
import time

# Initialize a local Spark session
spark = SparkSession.builder \
    .appName("Spark_Performance_Fundamentals") \
    .master("local[*]") \
    .getOrCreate()

print(f"Spark version: {spark.version}")

## Why Performance Matters

Apache Spark is designed to process large datasets across multiple machines. However:

- Adding more machines does not automatically make jobs fast.
- Poorly written Spark code can waste resources.
- Slow jobs increase cloud computing costs.
- Slow jobs delay critical business reports and analytics.

**Real World Scenario:**
Imagine an e-commerce company processing millions of daily orders. Management expects daily analytics reports. If a Spark job takes 5 minutes, that's excellent. If it takes 5 hours, you have a problem.

A Data Engineer's job is not only to make pipelines work, but to make them **efficient**.

## What Makes a Spark Job Slow?

Many beginners assume: *"Big data is large, therefore jobs are naturally slow."*

This is only partially true. Spark jobs usually become slow because of four major hardware bottlenecks:

1. **CPU Bottlenecks** (Compute)
2. **Memory Bottlenecks** (RAM)
3. **Disk Bottlenecks** (Storage I/O)
4. **Network Bottlenecks** (Data movement/Shuffle)

Let's simulate and study each one.

### Execution Flow Overview

Before identifying the bottleneck, remember the typical Spark execution flow:

`Data Source -> Read Data -> Transform Data -> Shuffle Data -> Aggregate/Join -> Write Results`

<h3>Execution Flow</h3>
<img src="../assets/Module_11/11_01_01.png" alt="Execution flow infographic" width="75%">
<p><i><b>Image Prompt:</b> Beginner-friendly Apache Spark execution pipeline diagram. Data Source → Read Data → Transformations → Shuffle → Aggregation/Join → Output Storage. Arrows showing data movement. Clean educational infographic, white background, modern training style.</i></p>

## 1. CPU Bottleneck

The CPU performs the actual computations (filtering, complex math, string parsing). When CPUs become overloaded, tasks sit in a queue waiting to be processed.

**Symptoms:**
- 100% CPU utilization
- Long task execution times even with small amounts of data

Let's simulate a CPU bottleneck by applying a heavy User Defined Function (UDF) that wastes CPU cycles without moving much data.

In [ ]:
# Simulating a CPU heavy operation
df_cpu = spark.range(1, 1000000)

# A UDF that intentionally burns CPU cycles (simulating poor code/complex regex)
@F.udf('long')
def heavy_computation(x):
    result = 0
    for i in range(100): # Waste CPU cycles
        result += i
    return x + result

print("Starting CPU intensive task...")
start_time = time.time()

# The action (count) triggers the heavy computation
df_cpu.withColumn("computed", heavy_computation(F.col("id"))).count()

print(f"Time taken: {time.time() - start_time:.2f} seconds")
print("Notice how time is spent processing, even though the dataset is relatively small.")

<h3>CPU Bottleneck</h3>
<img src="../assets/Module_11/11_01_02.png" alt="CPU Usage illustration" width="75%">
<p><i><b>Image Prompt:</b> Spark cluster with executors showing CPU usage reaching 100 percent. Tasks waiting in queue because processors are busy. Educational performance bottleneck diagram. White background.</i></p>

## 2. Memory Bottleneck & Disk Spill

Spark loves RAM. It processes data fastest when everything fits in memory. 

Problems occur when:
- Datasets explode in size during processing
- Executors have insufficient RAM
- Data skew causes one executor to get too much data

When memory fills up, Spark doesn't crash immediately; it temporarily writes the overflow to disk. This is called **Disk Spill**. Because disks are vastly slower than RAM, your job will suddenly crawl to a halt.

In [ ]:
# Simulating Memory Pressure (Data Explosion)
# A cross-join multiplies every row by every row. 
# 1,000 rows x 1,000 rows = 1,000,000 rows generated instantly in memory.

df_small = spark.range(1, 1000)

print("Starting memory-expanding task (Cross Join)...")
start_time = time.time()

df_memory = df_small.crossJoin(df_small)
row_count = df_memory.count()

print(f"Resulting rows from 1000x1000 cross join: {row_count:,}")
print(f"Time taken: {time.time() - start_time:.2f} seconds")
print("If we did this with 1 million rows, RAM would fill up immediately, forcing a slow Disk Spill.")

<h3>Memory Spill Example</h3>
<img src="../assets/Module_11/11_01_03.png" alt="Disk spill illustration" width="75%">
<p><i><b>Image Prompt:</b> Apache Spark executor memory full. Data overflowing from RAM into disk storage labeled 'Spill to Disk'. Performance slowdown illustration. Beginner-friendly infographic.</i></p>

## 3. Disk (I/O) Bottleneck

Spark constantly interacts with storage (reading CSV/Parquet, writing output). Slow storage systems, or writing in highly inefficient formats, drastically increases runtime.

A common beginner mistake is the **"Small Files Problem"**, where you write thousands of tiny files instead of a few large ones. This forces the disk driver to open and close thousands of connections, destroying performance.

In [ ]:
import shutil
import os

# Simulating an I/O Bottleneck: Writing 1000 tiny partitions to disk
df_io = spark.range(1, 10000).repartition(1000) # Forcing 1000 partitions/files

print("Writing 1000 tiny files to disk (I/O heavy)...")
start_time = time.time()
out_path = "/tmp/tiny_files_demo"

df_io.write.mode("overwrite").parquet(out_path)

print(f"Write time (1000 files): {time.time() - start_time:.2f} seconds")

# Cleanup
shutil.rmtree(out_path, ignore_errors=True)
print("Reading/Writing many tiny files forces massive disk I/O overhead compared to one large file.")

<h3>Disk I/O Bottleneck</h3>
<img src="../assets/Module_11/11_01_04.png" alt="I/O Bottleneck illustration" width="75%">
<p><i><b>Image Prompt:</b> Spark executors waiting while data slowly arrives from storage system. Disk icon connected to executors with slow arrows. I/O bottleneck visualization. White background.</i></p>

## 4. Network Bottleneck (The Shuffle)

Spark is a distributed system. Often, data needed for a calculation (like a group-by or a join) is spread across different machines.

To group all records belonging to "Customer A" together, Spark must move data across the network. This process is called a **Shuffle**. Shuffles are the most expensive operation in Spark because they involve Disk I/O, Network I/O, and CPU serialization.

In [ ]:
# Simulating a Network Shuffle
# We create dummy data and force a groupBy operation, which requires a shuffle.
df_shuffle = spark.range(1, 2000000).withColumn("random_key", (F.rand() * 10).cast("int"))

print("Starting shuffle intensive task...")
start_time = time.time()

# GroupBy forces Spark to shuffle data so identical keys end up on the same executor
df_shuffle.groupBy("random_key").count().show()

print(f"Time taken for shuffle and aggregation: {time.time() - start_time:.2f} seconds")

<h3>Network Shuffle</h3>
<img src="../assets/Module_11/11_01_05.png" alt="Network Shuffle illustration" width="75%">
<p><i><b>Image Prompt:</b> Apache Spark shuffle operation. Multiple executors exchanging data across the network before aggregation. Arrows between executors showing heavy data transfer. Educational architecture diagram.</i></p>

## Summary: The Four Major Bottlenecks

| Resource | Problem |
|-----------|----------|
| **CPU** | Too much computation, complex functions, bad code. |
| **Memory** | Insufficient RAM, large datasets exploding, forcing slow Disk Spills. |
| **Disk** | Slow read/write operations, small file problems. |
| **Network**| Excessive data movement (Shuffles) during Joins and GroupBys. |

Most Spark performance issues can be traced back to one of these categories.

---

## Why Throwing More Hardware Doesn't Always Help

Many beginners think: *"Let's just double the cluster size to make it faster."*

But this doesn't always solve the problem:
- Poor code remains poor code.
- Excessive shuffles remain expensive over the network.
- Bad file layouts still throttle disks.

Optimization usually provides significantly better (and cheaper) gains than blindly increasing cluster size.

## Key Takeaways

- Spark jobs become slow for specific hardware and architectural reasons.
- Distributed systems introduce network data movement costs (Shuffles).
- Disk spills (running out of RAM) drastically slow down pipelines.
- Understanding **where** time is spent is the first step toward optimization.

### Next Lesson: 11.2 Spark UI Deep Dive
We know the theory. Next, we will learn how to use the **Spark UI** to visually identify these exact CPU, Memory, and Shuffle bottlenecks in running jobs.